# Q2F | Predict Test Data

> F. Make predictions of the labels on the test data, using the trained model with chosen (best) hyperparameters. Summarize performance using the RMSE. Three teams with the lowest RMSE (who followed correctly all the previous steps) will receive bonus points.

# Answer Summary

**Predictions on the test data**
- Both saved models predict every test day from the 11 raw attributes.

**The trained models with the chosen hyperparameters**
- D's linear regression and E's polynomial regression, each loaded exactly as its notebook selected, trained on the training days and saved.

**RMSE**
- E's polynomial model misses by less than D's linear model: 1055.3 against 1081.2 rentals per day.
- Most test days fall in months 9 to 12, which no validation block covered.

**Final submission**
- E's polynomial regression (`polynomial_regression.joblib`), the saved model with the lower test RMSE.

**Previous steps followed**
- The test data is used once: nothing is refit or tuned after the test days are scored.

---

# Imports

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error

import Q2.consts as Q2_CONSTS
from consts import TEST_SIZE

from utils import read_csv_to_df, SPLIT

pd.set_option('display.float_format', '{:.1f}'.format)  # rentals

# Fetch Data

In [2]:
df = read_csv_to_df(path=Q2_CONSTS.DATA_PATH, parse_dates=["dteday"])
df = df.rename(columns=Q2_CONSTS.SHORT_NAMES)
_, test_df = SPLIT.chronological_split(df=df, date_col="Date", test_size=TEST_SIZE)

ATTRIBUTES = ['Temp', 'FeelsLike', 'Humidity', 'Wind', 'Year', 'IsHoliday', 'IsWorkday', 'Season', 'Month', 'Weekday', 'Weather']
X_test, y_test = test_df[ATTRIBUTES], test_df[Q2_CONSTS.TARGET]

print(f"{len(test_df)} test days, {test_df['Date'].min():%Y-%m-%d} to {test_df['Date'].max():%Y-%m-%d}")

147 test days, 2012-08-07 to 2012-12-31


# Method

- **Models:** the two models D and E selected and saved, loaded from `Q2/MODELS` as they are: D's linear regression (`linear_regression.joblib`) and E's polynomial regression (`polynomial_regression.joblib`). Each is one pipeline that takes the 11 raw attributes and builds its own columns, so nothing is fit on the test days.
- **Test days:** the last 20% of days, 2012-08-07 to 2012-12-31, starting the day after D's last validation day. So most test days fall in months 9 to 12, which no validation block covered.
- **Metric:** RMSE in rentals per day on every test day, the metric D and E used.
- **Test data used once:** each model's RMSE is reported as it comes. Neither score is used to change or refit a model.
- **Submission:** the saved model with the lower test RMSE is the final submission.

# Load the Models

In [3]:
MODELS = {  # the models D and E selected and saved, as they are
	'D: linear regression': joblib.load(Q2_CONSTS.MODEL_OUTPUT_DIR / 'linear_regression.joblib'),
	'E: polynomial regression': joblib.load(Q2_CONSTS.MODEL_OUTPUT_DIR / 'polynomial_regression.joblib'),
}

# Test RMSE

In [4]:
PREDICTIONS = {name: model.predict(X_test) for name, model in MODELS.items()}

RMSE_COLUMN = f'test rmse ({Q2_CONSTS.TARGET_UNIT})'
RESULTS = pd.DataFrame({RMSE_COLUMN: {name: mean_squared_error(y_test, predicted, squared=False) for name, predicted in PREDICTIONS.items()}})
RESULTS['submission'] = RESULTS.index == RESULTS[RMSE_COLUMN].idxmin()  # the lower test RMSE is the final submission
RESULTS

,test rmse (rentals per day),submission
D: linear regression,1081.2,False
E: polynomial regression,1055.3,True


### Notes / Observations

- E's polynomial model misses by less than D's linear model on the test days: 1055.3 against 1081.2 rentals per day.
- Most test days fall in months 9 to 12, which no validation block covered.